# Building Model Base Table
This notebook is backbone for understanding of data and assumptions made at the stage of building mbt table.

Purpose: mbt table is ready-to-use for a feautre engineering (encoding, interaction, typecasting, etc) for modeling.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")

In [5]:
%load_ext autoreload
%autoreload 2

from src.utils.load import load
from src.utils.data import tag_feature_map, add_suffix, valid_cols
from src.data.base_table import make_constant, build_flag

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
df = load("data/canonical/events.parquet")
schema = load("configs/schema.yaml")
profile = load("configs/feature_profile.yaml")
tags = tag_feature_map(profile)

## Inspect missing feature

In [7]:
missing_features = df.isna().sum()[df.isna().sum() > 0].index
missing_features

Index(['affected_uid', 'reported_by_uid', 'location_id', 'category_id',
       'subcategory_id', 'reported_symptom', 'asset_id', 'assigned_team_gid',
       'assigned_uid', 'root_cause_id', 'change_request_id', 'vendor_id',
       'caused_by_change_id', 'resolution_id', 'resolved_by_uid'],
      dtype='object')

In [8]:
# percentage of missing values by entries in data
missing_count_by_entries = df.isna().sum().sort_values(ascending=False)
missing_pct_by_entries = (100 * missing_count_by_entries / df.shape[0]).round(2).rename('%missing_by_entries')

# percentage of missing values by cases in data
missing_count_by_cases = df.isna().groupby(df['case_id']).any().sum().sort_values(ascending=False)
missing_pct_by_cases = (100 * missing_count_by_cases / df['case_id'].nunique()).round(2).rename("%missing_by_cases")

missing_pct_record = pd.concat([missing_pct_by_cases, missing_pct_by_entries], axis=1)
missing_pct_record.query("`%missing_by_cases` > 0 or `%missing_by_entries` > 0")

,%missing_by_cases,%missing_by_entries
caused_by_change_id,99.99,99.98
vendor_id,99.94,99.83
asset_id,99.80,99.69
change_request_id,99.61,99.30
root_cause_id,99.04,98.38
assigned_uid,35.15,19.40
reported_symptom,24.58,23.26
assigned_team_gid,10.19,2.95
reported_by_uid,1.01,0.97
resolution_id,0.43,0.50


In [9]:
missing_pct_record.query("`%missing_by_cases` > 99").index

Index(['caused_by_change_id', 'vendor_id', 'asset_id', 'change_request_id',
       'root_cause_id'],
      dtype='object')

The first 5 features are 99% nulls. 

* The null values in 'caused_by_change_id', 'change_request_id', 'root_cause_id' suggest the cases were not related to such relatd records
* Such cases did not caused by a change in IT sytem, and have no request for changing IT system so issue can be resolved, no IT problem is registered or known is related to the case.
--- from product and dataset information.

The 99% feels to sparse but they're genuine for a healthy business. 

Presence flag of such feature is capture for now to maintain the intent of feature.

## Handling (flag) 99% missing feature

In [10]:
tags = tag_feature_map(profile)
tags['sparse']

['caused_by_change_id',
 'change_request_id',
 'root_cause_id',
 'vendor_id',
 'asset_id']

In [11]:
add_suffix(tags['sparse'], "pflag", 2)

['caused_by_change_pflag',
 'change_request_pflag',
 'root_cause_pflag',
 'vendor_pflag',
 'asset_pflag']

In [12]:
sparse_cols = valid_cols(tags['sparse'], df)
df[add_suffix(sparse_cols, "pflag", 2)] = build_flag(df[sparse_cols], 
                                                          missing_flags=False)

In [13]:
# handled sparse value with flag 
df.drop(sparse_cols, axis=1, inplace=True)

## Inspect miniscule changing features

In [14]:
print(f"all {df.columns.size -1} features properties of 24_918 cases at case_level")
def generate_df_prop(df):
    grp = df.groupby('case_id')
    df_prop = pd.DataFrame(df.dtypes, columns=['dtype']).drop('case_id', axis=0)

    df_prop['no_of_unique'] = df.nunique(dropna=True)

    df_prop['no_of_changes'] = ((grp.nunique(dropna=True) <= 1).sum() - df.case_id.nunique(dropna=True)).round(2).abs()
    df_prop['pct_constant'] = (100* (grp.nunique(dropna=True) <= 1).sum() / df.case_id.nunique(dropna=True)).round(2).abs()

    df_prop['all_missing'] = df.isna().groupby(df['case_id']).all().sum()
    df_prop['any_missing'] = df.isna().groupby(df['case_id']).any().sum()
    return df_prop

df_prop = generate_df_prop(df)
df_prop.sort_values(['no_of_changes', 'any_missing'], ascending=True).query('no_of_changes < 10')

all 37 features properties of 24_918 cases at case_level


,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
opened_at,datetime64[ns],19849,0,100.00,0,0
created_at,datetime64[ns],19559,0,100.00,0,0
notify_email,boolean,2,0,100.00,0,0
resolved_at,datetime64[ns],19500,0,100.00,0,0
closed_at,datetime64[ns],2707,0,100.00,0,0
created_at_is_imputed,bool,2,0,100.00,0,0
affected_uid,object,5244,0,100.00,3,3
location_id,object,224,0,100.00,6,6
resolved_by_uid,object,216,0,100.00,99,99
resolution_id,object,17,0,100.00,107,107


contact_channel:
- 5 out of 25k cases making constant (case_leve) feature changing (event_level).
- assumption: intial contact_channel is used for communication and later channel update.
- The later channels might be used for further communication like escalation, sharing info. Since they're only 5 cases, model won't generalize on them.

In [15]:
df_prop.sort_values(['any_missing', 'no_of_changes'], ascending=True).query('any_missing > 0')

,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
affected_uid,object,5244,0,100.00,3,3
location_id,object,224,0,100.00,6,6
category_id,object,58,1184,95.25,7,7
subcategory_id,object,254,1738,93.03,8,8
resolved_by_uid,object,216,0,100.00,99,99
resolution_id,object,17,0,100.00,107,107
reported_by_uid,object,209,0,100.00,251,251
assigned_team_gid,object,78,9708,61.04,1,2539
reported_symptom,object,525,1323,94.69,5513,6126
assigned_uid,object,234,2858,88.53,658,8759


## Handling (drop) feature with minor missing cases


In [16]:
missing_table = df.isna().groupby(df['case_id']).all()
missing_count = missing_table.sum(axis=0)
feature_names = missing_count[(missing_count > 0) & (missing_count < 10)].index
feature_names

Index(['affected_uid', 'location_id', 'category_id', 'subcategory_id',
       'assigned_team_gid'],
      dtype='object')

In [17]:
cases_to_drop = df.isna().groupby(df['case_id']).all()[feature_names].any(axis=1)
drop_ids = cases_to_drop[cases_to_drop].index

print(df[~df['case_id'].isin(drop_ids)].shape)

(141647, 38)


In [18]:
df = df[~df['case_id'].isin(drop_ids)]

In [19]:
print(df.shape, "original data shape")

(141647, 38) original data shape


## Handling (fill) featues missing major (1000s) cases

In [20]:
missing_table = df.isna().groupby(df['case_id']).any()
missing_count = missing_table.sum(axis=0)
feature_names = missing_count[(missing_count > 1000)].index
feature_names

Index(['reported_symptom', 'assigned_team_gid', 'assigned_uid'], dtype='object')

In [21]:
(df[['reported_symptom', 'assigned_team_gid', 'assigned_uid']].isna()
 .groupby(df['case_id']).any().sum())

reported_symptom     6117
assigned_team_gid    2534
assigned_uid         8751
dtype: int64

In [22]:
(df[['reported_symptom', 'assigned_team_gid', 'assigned_uid']].isna()
 .groupby(df['case_id']).all().sum())

reported_symptom     5505
assigned_team_gid       0
assigned_uid          651
dtype: int64

In [23]:
df['reassignment_count'].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27])

Checking assumption that the assigned agent changes with every reassignment count (or maybe along with reopen_count)

In [24]:
df.groupby(['case_id', 'reassignment_count'])['assigned_uid'].nunique(dropna=True).value_counts()

assigned_uid
1    38217
0     8361
2     1668
3       60
4        7
5        1
Name: count, dtype: int64

In [25]:
df.groupby(['case_id', 'reopen_count', 'reassignment_count'])['assigned_uid'].nunique(dropna=True).value_counts()

assigned_uid
1    38529
0     8366
2     1683
3       58
4        7
5        1
Name: count, dtype: int64

The number of uniques assigned agent more than 1 conclude that the previous assumption doesn't hold in the dataset..

In [26]:
# cardinality check
df[feature_names].nunique()

reported_symptom     525
assigned_team_gid     78
assigned_uid         233
dtype: int64

'reported_symptom', 'assigned_team_gid', 'assigned_uid' are not highly missing values, and no valid product logic (reassignment/reopen count, features nmi) is found. 

- Forward or backward filling on assigned team or agent id will fabricate their pattern for model learning. This can be handled only by imputing "Unknown" value.
- reported_symptom is not a proper low cardinality (2% of the total cases), missing_flag likely to work best for this after forward. Assuming once the user has provided symptom/perception of issue it is known to the system/agent.

In [27]:
df[add_suffix('reported_symptom', "_mflag")] = build_flag(df['reported_symptom'], missing_flags=True) # capture original missingness
df['reported_symptom'] = df.groupby("case_id")['reported_symptom'].transform('ffill')
df['reported_symptom'] = df['reported_symptom'].fillna('Unknown')

In [28]:
df['assigned_uid'] = df['assigned_uid'].fillna('Unknown')
df['assigned_team_gid'] = df['assigned_team_gid'].fillna('Unknown')

## Inspect features missing 100s of cases

In [29]:
missing_table = df.isna().groupby(df['case_id']).all()
missing_count = missing_table.sum(axis=0)
feature_names = missing_count[(missing_count > 10) & (missing_count < 1000)].index.tolist()
feature_names

['reported_by_uid', 'resolution_id', 'resolved_by_uid']

resolution_id and resolved_by_uid columns are future values and will cause data leakage and are forbidden as model learning feature. 

In [30]:
df[feature_names].nunique()

reported_by_uid    208
resolution_id       17
resolved_by_uid    215
dtype: int64

### Handling (fill) 100s missing uid

In [31]:
tags['uid']

['affected_uid',
 'reported_by_uid',
 'resolved_by_uid',
 'assigned_team_gid',
 'assigned_uid']

In [32]:
valid_cols(tags['uid'], df)

['affected_uid',
 'reported_by_uid',
 'resolved_by_uid',
 'assigned_team_gid',
 'assigned_uid']

In [33]:
df_prop = generate_df_prop(df)
df_prop.loc[['affected_uid', 'reported_by_uid', 'resolved_by_uid', 'assigned_team_gid', 'assigned_uid']]

,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
affected_uid,object,5244,0,100.00,0,0
reported_by_uid,object,208,0,100.00,250,250
resolved_by_uid,object,215,0,100.00,98,98
assigned_team_gid,object,79,10944,56.06,0,0
assigned_uid,object,234,8947,64.08,0,0


reported_by_uid and assigned_uid are low-medium cardinality (200s in 25k cases) and uid does have any ordering. 


Note: missing affected_uid, assigned uid, and assigned_team_gid are already handld and thus now 0.
- such cases with missing affected_uid are dropped [## Handling (drop) feature with minor missing cases]
- assigned uid and gid are filled with unknown. [## Handling (fill) featues missing major (1000s) cases]

In [34]:
df[valid_cols(tags['uid'], df)] = df[valid_cols(tags['uid'], df)].fillna("Unknown")

## Handling features - minor changing within cases.

In [35]:
df_prop = generate_df_prop(df)
df_prop.sort_values(['no_of_changes', 'any_missing'], ascending=True).query('no_of_changes > 0 and pct_constant > 50')

,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
caused_by_change_pflag,int64,2,1,100.00,0,0
contact_channel,object,5,5,99.98,0,0
asset_pflag,int64,2,5,99.98,0,0
vendor_pflag,int64,2,52,99.79,0,0
change_request_pflag,int64,2,82,99.67,0,0
root_cause_pflag,int64,2,143,99.43,0,0
used_knowledge_base,bool,2,208,99.16,0,0
reopen_count,int64,9,275,98.90,0,0
urgency_level,object,3,299,98.80,0,0
impact_level,object,3,315,98.74,0,0


Flags are meant to capture the intent from original data, they won't be made case_level constant

Following features are likely genuine change and real signal:
- used_knowledge_base: ex, first knowledge not used, later accessed for further investigation.
- reopen_count: only 1% cases are reopened. 
- priority_level: ex, user raised the urgency later to reduce deadline; agent changed impact upon case investigation. 
- reopen_count is likely weak feature for model.

similar senarios can be listed for others in the list.

## Inspect transition of features 10% changing (>90% constant).

In [36]:
def summarize_transitions(df, col):
    nunique = df.groupby('case_id')[col].nunique()
    varying_ids = nunique[nunique > 1].index  # pre-filter
    
    return (
        df[df['case_id'].isin(varying_ids)]
        .sort_values(['case_id', 'updated_at', 'system_update_count'])
        .groupby('case_id')[col]
        .apply(lambda x: f"{x.iloc[0]} → {x.iloc[-2]}") # last event is mostly Closed status
        .value_counts()
    )

In [37]:
summarize_transitions(df, 'used_knowledge_base')

used_knowledge_base
True → False    106
False → True    102
Name: count, dtype: int64

Both true to false, and false to true have almost same distribution. 

Because they are changing in only 1% of cases. For this, case_level constant proxy to be created using first value from the cases.

In [38]:
summarize_transitions(df, 'urgency_level')

urgency_level
2 - Medium → 1 - High      195
1 - High → 2 - Medium       48
2 - Medium → 3 - Low        18
1 - High → 3 - Low          14
2 - Medium → 2 - Medium     13
3 - Low → 2 - Medium         6
3 - Low → 1 - High           3
1 - High → 1 - High          2
Name: count, dtype: int64

In [39]:
summarize_transitions(df, 'impact_level')

impact_level
2 - Medium → 1 - High      194
1 - High → 2 - Medium       51
2 - Medium → 3 - Low        24
2 - Medium → 2 - Medium     21
1 - High → 3 - Low          12
3 - Low → 2 - Medium         7
3 - Low → 1 - High           4
1 - High → 1 - High          2
Name: count, dtype: int64

In [40]:
summarize_transitions(df, 'priority_level')

priority_level
3 - Moderate → 2 - High        123
3 - Moderate → 1 - Critical    123
1 - Critical → 3 - Moderate     35
3 - Moderate → 4 - Low          22
1 - Critical → 4 - Low          18
2 - High → 1 - Critical         16
3 - Moderate → 3 - Moderate     13
2 - High → 3 - Moderate         11
4 - Low → 3 - Moderate           7
1 - Critical → 2 - High          5
4 - Low → 1 - Critical           4
2 - High → 4 - Low               4
1 - Critical → 1 - Critical      2
Name: count, dtype: int64

The adjustment in priority (urgency/impact) level was performed majorly to escalate the case.


## Handle <10% changing features - constant proxy

convert minor changing features so model can learn that these are constant case_level feature. 

first event data will be used to convert feature to constant case_level.

Modeling will be experimented with both - original and constant-proxy features.

In [47]:
df_prop.query('100 > pct_constant > 90').sort_values('pct_constant')

,dtype,no_of_unique,no_of_changes,pct_constant,all_missing,any_missing
subcategory_id,object,254,1737,93.03,0,0
reported_symptom,object,526,1531,93.85,0,0
category_id,object,57,1182,95.25,0,0
reported_symptom_mflag,int64,2,612,97.54,0,0
priority_level,object,4,383,98.46,0,0
impact_level,object,3,315,98.74,0,0
urgency_level,object,3,299,98.80,0,0
reopen_count,int64,9,275,98.90,0,0
used_knowledge_base,bool,2,208,99.16,0,0
root_cause_pflag,int64,2,143,99.43,0,0


In [ ]:
# removed flag as it capture original data
minor_changing_feature = valid_cols(list(set(tags['minor_change']) - set(tags['flag'])), df)
minor_changing_feature

cproxy_feature = add_suffix(minor_changing_feature, "_cproxy") #constant proxy features
cproxy_feature

['used_knowledge_base_cproxy',
 'reported_symptom_cproxy',
 'impact_level_cproxy',
 'subcategory_id_cproxy',
 'category_id_cproxy',
 'contact_channel_cproxy',
 'priority_level_cproxy',
 'urgency_level_cproxy']

In [64]:
df[cproxy_feature] = make_constant(df, minor_changing_feature)
df.shape

(141647, 47)

In [65]:
(df[cproxy_feature].groupby(df['case_id']).nunique() > 1).sum()

used_knowledge_base_cproxy    0
reported_symptom_cproxy       0
impact_level_cproxy           0
subcategory_id_cproxy         0
category_id_cproxy            0
contact_channel_cproxy        0
priority_level_cproxy         0
urgency_level_cproxy          0
dtype: int64